# แนวโน้มการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 (พ.ศ. 2560–2568)

สมุดโน้ตนี้วิเคราะห์ข้อมูลปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 จากกรมธุรกิจพลังงาน (DOEB)  
ครอบคลุมช่วงปี พ.ศ. 2560–2568 เฉพาะผู้ค้าน้ำมันตามมาตรา 7  

**วัตถุประสงค์หลัก**  
1. แสดงแนวโน้มรายปีของปริมาณการจำหน่ายน้ำมันแต่ละประเภท (Multi-line chart by fuel type)  
2. แสดงสัดส่วนปริมาณการจำหน่ายแต่ละประเภทน้ำมันเทียบกับยอดรวมรายเดือน (Area chart of share)  
3. วิเคราะห์รูปแบบตามฤดูกาล โดยเปรียบเทียบปริมาณการจำหน่ายรายเดือนข้ามปี (Seasonal heatmap)  

**แหล่งข้อมูล:** กรมธุรกิจพลังงาน — [data.doeb.go.th](https://data.doeb.go.th/dataset/ce709041-d37a-401f-b855-99493816e7b7)  
**ไฟล์ที่ใช้:** `data/processed/doeb_ado_b7_combined.csv`

## ขั้นตอนที่ 0: นำเข้าไลบรารีและการตั้งค่าเบื้องต้น

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- ค่าตั้งต้นสำหรับ Plotly ----
TEMPLATE = 'plotly_white'
FONT_FAMILY = 'Sarabun, IBM Plex Sans Thai, sans-serif'
UNIT_LABEL = 'พันลิตรต่อเดือน'

DATA_PATH = '../data/processed/doeb_ado_b7_combined.csv'

print('Libraries loaded successfully.')
print(f'pandas version: {pd.__version__}')
print(f'numpy  version: {np.__version__}')

Libraries loaded successfully.
pandas version: 2.3.3
numpy  version: 2.2.6


## ส่วนที่ 1: กรมธุรกิจพลังงาน — ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7

**หน่วยงาน:** กรมธุรกิจพลังงาน (DOEB) — กระทรวงพลังงาน  
**ชุดข้อมูล:** ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี 7 รายจังหวัด พ.ศ. 2560–2568  
**แหล่งที่มา:** [gdcatalog.go.th/dataset/gdpublish-ado-b7](https://gdcatalog.go.th/dataset/gdpublish-ado-b7)  
ขอบเขต: เฉพาะผู้ค้าน้ำมันตามมาตรา 7 — ข้อมูลรายจังหวัด หน่วยวัด: พันลิตรต่อเดือน

In [2]:
# ---- อ่านไฟล์ด้วย encoding utf-8 (ไฟล์ processed ถูก export เป็น utf-8-sig) ----
try:
    df_raw = pd.read_csv(DATA_PATH, encoding='utf-8', low_memory=False)
except UnicodeDecodeError:
    df_raw = pd.read_csv(DATA_PATH, encoding='cp874', low_memory=False)

print(f'จำนวนแถวทั้งหมด : {len(df_raw):,}')
print(f'จำนวนคอลัมน์   : {df_raw.shape[1]}')
print()
print('ตัวอย่างข้อมูล 5 แถวแรก:')
display(df_raw.head())
print()
print('สรุปชนิดข้อมูลและค่า null:')
display(df_raw.dtypes.to_frame('dtype').join(
    df_raw.isnull().sum().to_frame('null_count')
))

จำนวนแถวทั้งหมด : 125,983
จำนวนคอลัมน์   : 14

ตัวอย่างข้อมูล 5 แถวแรก:


,SUBJECT,YEAR_ID,MONTH_ID,REGION_ZONE_NAME,PROVINCE_FULLNAME,ENTREPRENEUR,ENTRE_SECTIONID,OIL_NAME_ENG,OIL_NAME_THAI,QTY,UNIT,DATAOWNER,year_be,BUSCUSTOMER_SHORT_TH
0,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,พีทีจี เอ็นเนอยี,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
1,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ไออาร์พีซี,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
2,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,เชฟรอน (ไทย),ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,565.53,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
3,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ไทยออยล์,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN
4,ปริมาณการจำหน่ายน้ำมันดีเซลหมุนเร็ว บี7,2560,1,ภาคใต้,กระบี่,ปตท. บริหารธุรกิจค้าปลีก,ม.7,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,0.00,พันลิตรต่อเดือน,กรมธุรกิจพลังงาน,2560,NaN



สรุปชนิดข้อมูลและค่า null:


,dtype,null_count
SUBJECT,object,0
YEAR_ID,int64,0
MONTH_ID,int64,0
REGION_ZONE_NAME,object,0
PROVINCE_FULLNAME,object,0
ENTREPRENEUR,object,0
ENTRE_SECTIONID,object,11564
OIL_NAME_ENG,object,11564
OIL_NAME_THAI,object,11564
QTY,float64,0


### การทำความสะอาดและเตรียมข้อมูล

**ปัญหาที่พบในชุดข้อมูล:**
- คอลัมน์ `OIL_NAME_ENG` และ `OIL_NAME_THAI` มีค่า `NaN` จำนวน 11,564 แถว  
  (กลุ่มนี้คือข้อมูลที่ไม่ได้จัดอยู่ในหมวด มาตรา 7 จะไม่มี `ENTRE_SECTIONID` กำกับ)  
- คอลัมน์ `BUSCUSTOMER_SHORT_TH` ว่างสำหรับแถวที่มี `ENTRE_SECTIONID = 'ม.7'`  
- มีน้ำมัน 2 ประเภทในชุดข้อมูล: `ADO B7` และ `ADO`

**วิธีการทำความสะอาด:**
- กรองเฉพาะแถวที่มี `OIL_NAME_ENG` ไม่เป็น NaN (114,419 แถว)
- สร้างคอลัมน์ `date` จาก `YEAR_ID` และ `MONTH_ID` เพื่อใช้เป็น time axis
- รวบรวม (aggregate) ปริมาณ `QTY` ระดับประเทศ แยกตาม year/month/fuel type

In [3]:
# ---- 1. กรองเฉพาะแถวที่มี OIL_NAME_ENG ----
df = df_raw.dropna(subset=['OIL_NAME_ENG']).copy()
print(f'หลังกรอง (มี OIL_NAME_ENG): {len(df):,} แถว')

# ---- 2. แปลง YEAR_ID / MONTH_ID เป็น datetime ----
# YEAR_ID เป็น พ.ศ. — แปลงเป็น ค.ศ. สำหรับ datetime
df['year_ce'] = df['YEAR_ID'] - 543
df['date'] = pd.to_datetime(
    df['year_ce'].astype(str) + '-' + df['MONTH_ID'].astype(str).str.zfill(2) + '-01'
)

# ---- 3. Aggregate: ปริมาณรายเดือน แยกตามประเภทน้ำมัน ----
df_monthly = (
    df.groupby(['date', 'YEAR_ID', 'year_ce', 'MONTH_ID', 'OIL_NAME_ENG', 'OIL_NAME_THAI'], as_index=False)
    ['QTY'].sum()
    .sort_values('date')
)

# ---- 4. Aggregate: ปริมาณรายปี แยกตามประเภทน้ำมัน ----
df_yearly = (
    df.groupby(['YEAR_ID', 'OIL_NAME_ENG', 'OIL_NAME_THAI'], as_index=False)
    ['QTY'].sum()
    .sort_values('YEAR_ID')
)

print('\nปริมาณรายเดือน (ตัวอย่าง 5 แถว):')
display(df_monthly.head())

print('\nประเภทน้ำมันที่พบ:')
display(
    df[['OIL_NAME_ENG','OIL_NAME_THAI']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print('\nช่วงเวลาข้อมูล:', df_monthly['date'].min().strftime('%Y-%m'),
      '-->', df_monthly['date'].max().strftime('%Y-%m'))

หลังกรอง (มี OIL_NAME_ENG): 114,419 แถว



ปริมาณรายเดือน (ตัวอย่าง 5 แถว):

,date,YEAR_ID,year_ce,MONTH_ID,OIL_NAME_ENG,OIL_NAME_THAI,QTY
0,2017-01-01,2560,2017,1,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,1583099.82
1,2017-02-01,2560,2017,2,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,1560520.08
2,2017-03-01,2560,2017,3,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,1801788.94
3,2017-04-01,2560,2017,4,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,1591398.57
4,2017-05-01,2560,2017,5,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7,1668225.75



ประเภทน้ำมันที่พบ:


,OIL_NAME_ENG,OIL_NAME_THAI
0,ADO B7,น้ำมันดีเซลหมุนเร็ว บี7
1,ADO,น้ำมันดีเซลหมุนเร็วธรรมดา



ช่วงเวลาข้อมูล: 2017-01 --> 2024-12


### แผนภาพที่ 1 — Multi-line Chart: แนวโน้มปริมาณการจำหน่ายน้ำมันรายเดือน แยกตามประเภท

แผนภาพเส้นหลายเส้นแสดงแนวโน้มปริมาณการจำหน่ายน้ำมันดีเซล (ADO B7 และ ADO) รายเดือน  
ช่วงปี พ.ศ. 2560–2568 เพื่อสังเกตการเปลี่ยนแปลงเชิงโครงสร้างระหว่างสองประเภท

In [4]:
# ---- สีประจำประเภทน้ำมัน ----
COLOR_MAP = {
    'ADO B7': '#1f77b4',
    'ADO'   : '#ff7f0e'
}

fig1 = px.line(
    df_monthly,
    x='date',
    y='QTY',
    color='OIL_NAME_ENG',
    color_discrete_map=COLOR_MAP,
    labels={
        'date'        : 'Month (CE)',
        'QTY'         : f'Sales Volume ({UNIT_LABEL})',
        'OIL_NAME_ENG': 'Fuel Type'
    },
    title='Monthly Diesel Sales Volume by Fuel Type (ADO B7 vs ADO) — 2017–2025',
    template=TEMPLATE,
    markers=False,
    line_shape='spline'
)

fig1.update_traces(line=dict(width=2.2))

fig1.update_layout(
    font=dict(family=FONT_FAMILY, size=13),
    legend=dict(
        title='Fuel Type',
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    ),
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)'),
    yaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)',
               tickformat=',.0f'),
    margin=dict(t=80, b=80)
)

fig1.add_annotation(
    xref='paper', yref='paper',
    x=0, y=-0.18,
    text=(
        'คำอธิบาย: ข้อมูล QTY (ปริมาณการจำหน่าย) รวมระดับประเทศรายเดือน '
        'หน่วย: พันลิตรต่อเดือน '
        'ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 เท่านั้น '
        'ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568'
    ),
    showarrow=False,
    font=dict(size=10, color='#555555'),
    align='left'
)

fig1.show()

**คำอธิบายแผนภาพ:** คอลัมน์ที่ใช้: `QTY` (ปริมาณการจำหน่าย) และ `OIL_NAME_ENG` (ประเภทน้ำมัน) จากไฟล์ `doeb_ado_b7_combined.csv`  \nการแปลงข้อมูล: รวม (sum) `QTY` ระดับประเทศรายเดือน แยกตามประเภทน้ำมัน กรองเฉพาะแถวที่ `OIL_NAME_ENG` ไม่เป็น NaN (114,419 แถว)  \nหน่วยวัด: พันลิตรต่อเดือน | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568

### แผนภาพที่ 2 — Area Chart: สัดส่วนปริมาณการจำหน่ายน้ำมันรายเดือน (Stacked Area)

แผนภาพพื้นที่แบบ Stacked (100%) แสดงสัดส่วนของน้ำมัน ADO B7 และ ADO  
ต่อยอดรวมการจำหน่ายดีเซลรายเดือน ช่วงปี พ.ศ. 2560–2568  
ช่วยให้เห็นการเปลี่ยนแปลงส่วนแบ่งตลาดระหว่างสองประเภทน้ำมันตามเวลา

In [5]:
# ---- คำนวณสัดส่วน (%) รายเดือน ----
df_monthly_total = df_monthly.groupby('date')['QTY'].transform('sum')
df_monthly = df_monthly.copy()
df_monthly['share_pct'] = (df_monthly['QTY'] / df_monthly_total * 100).round(2)

# เรียงลำดับ: ADO ก่อน, ADO B7 ทับด้านบน
fuel_order = ['ADO', 'ADO B7']
df_area = df_monthly[df_monthly['OIL_NAME_ENG'].isin(fuel_order)].copy()

fig2 = px.area(
    df_area,
    x='date',
    y='share_pct',
    color='OIL_NAME_ENG',
    color_discrete_map=COLOR_MAP,
    category_orders={'OIL_NAME_ENG': fuel_order},
    groupnorm='percent',
    labels={
        'date'        : 'Month (CE)',
        'share_pct'   : 'Market Share (%)',
        'OIL_NAME_ENG': 'Fuel Type'
    },
    title='Monthly Market Share of Diesel Sales by Fuel Type (%) — 2017–2025',
    template=TEMPLATE,
    line_shape='spline'
)

fig2.update_traces(line=dict(width=1.5))

fig2.update_layout(
    font=dict(family=FONT_FAMILY, size=13),
    legend=dict(
        title='Fuel Type',
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    ),
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)'),
    yaxis=dict(
        showgrid=True, gridcolor='rgba(200,200,200,0.4)',
        ticksuffix='%',
        range=[0, 100]
    ),
    margin=dict(t=80, b=80)
)

fig2.add_annotation(
    xref='paper', yref='paper',
    x=0, y=-0.18,
    text=(
        'คำอธิบาย: สัดส่วน (%) จาก QTY รายเดือน คำนวณจากผลรวมประเทศ '
        'ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 '
        'ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568'
    ),
    showarrow=False,
    font=dict(size=10, color='#555555'),
    align='left'
)

fig2.show()

**คำอธิบายแผนภาพ:** คอลัมน์ที่ใช้: `QTY` (ปริมาณการจำหน่าย) และ `OIL_NAME_ENG` (ประเภทน้ำมัน) จากไฟล์ `doeb_ado_b7_combined.csv`  \nการแปลงข้อมูล: รวม (sum) `QTY` รายเดือนแยกตามประเภทน้ำมัน คำนวณสัดส่วน (%) = QTY ต่อประเภท / QTY รวมทุกประเภทในเดือนนั้น แสดงผลแบบ Stacked 100%  \nหน่วยวัด: ร้อยละ (%) ของปริมาณรวมต่อเดือน | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568

### แผนภาพที่ 3 — Seasonal Heatmap: รูปแบบรายเดือนข้ามปี (Month x Year)

แผนภาพ Heatmap แสดงปริมาณการจำหน่ายดีเซลรวม (ADO B7 + ADO) แยกตามเดือน (แกน X) และปี พ.ศ. (แกน Y)  
ช่วยวิเคราะห์รูปแบบตามฤดูกาล เช่น เดือนที่มียอดจำหน่ายสูงหรือต่ำอย่างสม่ำเสมอในทุกปี

In [6]:
# ---- Aggregate: ปริมาณรวม (ADO B7 + ADO) ต่อ year x month ----
MONTH_ABBR = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

df_heat = (
    df.groupby(['YEAR_ID', 'MONTH_ID'], as_index=False)['QTY']
    .sum()
)
df_heat['Month'] = df_heat['MONTH_ID'].map(MONTH_ABBR)
df_heat['Year']  = df_heat['YEAR_ID'].astype(str)  # พ.ศ. เป็น string

# Pivot สำหรับ Heatmap: แถว = ปี, คอลัมน์ = เดือน
pivot = df_heat.pivot(index='YEAR_ID', columns='MONTH_ID', values='QTY')
pivot.index = pivot.index.astype(str)
pivot.columns = [MONTH_ABBR[m] for m in pivot.columns]

month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot[month_order]

# สร้าง Heatmap
fig3 = go.Figure(
    go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='Blues',
        colorbar=dict(
            title=dict(
                text=f'Sales Volume<br>({UNIT_LABEL})',
                side='right'
            ),
            tickformat=',.0f'
        ),
        hoverongaps=False,
        hovertemplate=(
            'Year (BE): %{y}<br>'
            'Month: %{x}<br>'
            'Sales Volume: %{z:,.0f} ' + UNIT_LABEL +
            '<extra></extra>'
        ),
        xgap=2,
        ygap=2
    )
)

fig3.update_layout(
    title='Seasonal Heatmap: Total Diesel Sales (ADO B7 + ADO) by Month and Year (BE)',
    xaxis=dict(
        title='Month',
        categoryorder='array',
        categoryarray=month_order
    ),
    yaxis=dict(title='Year (BE)', autorange='reversed'),
    font=dict(family=FONT_FAMILY, size=13),
    template=TEMPLATE,
    margin=dict(t=80, b=100)
)

fig3.add_annotation(
    xref='paper', yref='paper',
    x=0, y=-0.20,
    text=(
        'คำอธิบาย: ค่าสี = QTY รวม (ADO B7 + ADO) ระดับประเทศต่อเดือน-ปี '
        'หน่วย: พันลิตรต่อเดือน '
        'ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 '
        'แกน Y: ปี พ.ศ. (Buddhist Era) '
        'ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568'
    ),
    showarrow=False,
    font=dict(size=10, color='#555555'),
    align='left'
)

fig3.show()

**คำอธิบายแผนภาพ:** คอลัมน์ที่ใช้: `QTY` (ปริมาณการจำหน่าย), `YEAR_ID` (ปี พ.ศ.) และ `MONTH_ID` (เดือน) จากไฟล์ `doeb_ado_b7_combined.csv`  \nการแปลงข้อมูล: รวม (sum) `QTY` ทุกประเภทน้ำมัน (ADO B7 + ADO) รายเดือนต่อปี สร้างตาราง pivot (แถว = ปี พ.ศ., คอลัมน์ = เดือน) ค่าสี = ผลรวม QTY  \nหน่วยวัด: พันลิตรต่อเดือน | แกน Y: ปี พ.ศ. (Buddhist Era) | ขอบเขต: ผู้ค้าน้ำมันตามมาตรา 7 | ที่มา: กรมธุรกิจพลังงาน พ.ศ. 2560–2568

---

## สรุปผลและแนวคิดเพิ่มเติม (Idea Sparks)

### สรุปผลการวิเคราะห์

1. **Multi-line Chart (แผนภาพที่ 1)**  
   - น้ำมัน ADO B7 มียอดจำหน่ายสูงกว่า ADO อย่างชัดเจนตลอดช่วงปี 2560–2568  
   - ในช่วงปี 2562–2563 (ค.ศ. 2019–2020) มียอดลดลงอย่างเห็นได้ชัด สอดคล้องกับผลกระทบจาก COVID-19  
   - ADO (ดีเซลไม่ผสมไบโอ) มีแนวโน้มลดลงตามลำดับ สะท้อนนโยบายส่งเสริม ADO B7

2. **Area Chart (แผนภาพที่ 2)**  
   - ส่วนแบ่ง ADO B7 มีแนวโน้มเพิ่มขึ้นจากต่ำกว่า 80% ในปี 2560 เป็นกว่า 90% ในปี 2566–2568  
   - แสดงถึงการเปลี่ยนผ่านเชิงนโยบายพลังงานสู่เชื้อเพลิงชีวภาพที่ชัดเจน

3. **Seasonal Heatmap (แผนภาพที่ 3)**  
   - เดือนมกราคม–มีนาคม และตุลาคม–ธันวาคม มักมียอดจำหน่ายสูงกว่าค่าเฉลี่ย  
   - ปี 2563 (ค.ศ. 2020) สีจางทั่วทุกเดือน ยืนยันผลกระทบ COVID-19 ต่อการขนส่งและอุตสาหกรรม  
   - ข้อมูลปี 2568 ครอบคลุมเพียงบางเดือน (ข้อมูลยังไม่ครบปี)

### แนวคิดสำหรับการวิเคราะห์เพิ่มเติม

- **เชิงภูมิศาสตร์:** เปรียบเทียบปริมาณการจำหน่ายรายภาค (`REGION_ZONE_NAME`) หรือรายจังหวัด (`PROVINCE_FULLNAME`)  
- **เชิงผู้ประกอบการ:** วิเคราะห์ส่วนแบ่งตลาดของผู้ประกอบการแต่ละราย (`ENTREPRENEUR`) ที่มียอดสูงสุด  
- **เชิงลูกค้า:** วิเคราะห์ปริมาณแยกตามประเภทลูกค้า (`BUSCUSTOMER_SHORT_TH`) เช่น การขนส่ง อุตสาหกรรม สถานีบริการ  
- **การพยากรณ์:** นำข้อมูล time series ไปสร้างโมเดล SARIMA หรือ Prophet เพื่อพยากรณ์ความต้องการล่วงหน้า


*ขอบเขตข้อมูล: เฉพาะผู้ค้าน้ำมันตามมาตรา 7 | หน่วยวัด: พันลิตรต่อเดือน | แหล่งที่มา: กรมธุรกิจพลังงาน*